In [1]:
import os
import json
import numpy as np
import pandas as pd

import rasterio
from rasterio.transform import from_origin
import gc

In [ ]:
maize_env = pd.read_csv('../../dataset/region prediction/maize_soil_all.csv')
wheat_env = pd.read_csv('../../dataset/region prediction/wheat_soil_all.csv')
drop_cols = ['crop_type']
maize_env = maize_env.drop(
    columns=drop_cols,
    errors='ignore'
)

wheat_env = wheat_env.drop(
    columns=drop_cols,
    errors='ignore'
)

In [10]:
print(maize_env["lon"].min(), maize_env["lon"].max())
print(maize_env["lat"].min(), maize_env["lat"].max())

74.78407612583459 134.5604024350057
18.325153199763644 52.4080825282826


In [ ]:
n = len(maize_env)

lon_range = maize_env["lon"].max() - maize_env["lon"].min()
lat_range = maize_env["lat"].max() - maize_env["lat"].min()

#print("average grid size:", np.sqrt(lon_range * lat_range / n))

平均经度间距: 0.03463732969190789


In [ ]:
for res in [0.05, 0.01, 0.005, 0.001]:

    lon_q = np.round(maize_env["lon"] / res) * res
    lat_q = np.round(maize_env["lat"] / res) * res

    n_grid = len(
        pd.DataFrame({
            "lon": lon_q,
            "lat": lat_q
        }).drop_duplicates()
    )

    print(
        f"resolution={res:<6}",
        f"number of only grids={n_grid:<8}",
        f"compression ratio={n / n_grid:.2f}"
    )

resolution=0.05   唯一网格数=94868    压缩率=17.90
resolution=0.01   唯一网格数=819951   压缩率=2.07
resolution=0.005  唯一网格数=1596502  压缩率=1.06
resolution=0.001  唯一网格数=1698155  压缩率=1.00


In [ ]:
# ==========================
# Point to raster
# ==========================

def points_to_raster(
    df,
    value_col,
    output_tif,
    resolution=0.05
):

    df = df[['lon', 'lat', value_col]].copy()

    #  to 0.05° grid
    df['lon'] = (
        np.round(df['lon'] / resolution)
        * resolution
    )

    df['lat'] = (
        np.round(df['lat'] / resolution)
        * resolution
    )

    df = (
        df.groupby(
            ['lon', 'lat'],
            as_index=False
        )[value_col]
        .mean()
    )

    print(f'\n{value_col}')
    print(f'Grid cells: {len(df):,}')

    xmin = df['lon'].min()
    xmax = df['lon'].max()

    ymin = df['lat'].min()
    ymax = df['lat'].max()

    ncols = int(
        round((xmax - xmin) / resolution)
    ) + 1

    nrows = int(
        round((ymax - ymin) / resolution)
    ) + 1

    print(f'Raster size: {nrows} × {ncols}')

    nodata = -9999.0

    raster = np.full(
        (nrows, ncols),
        nodata,
        dtype=np.float32
    )

    cols = (
        (df['lon'] - xmin) / resolution
    ).round().astype(int)

    rows = (
        (ymax - df['lat']) / resolution
    ).round().astype(int)

    raster[
        rows,
        cols
    ] = df[value_col].values

    transform = from_origin(
        xmin - resolution / 2,
        ymax + resolution / 2,
        resolution,
        resolution
    )

    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=nrows,
        width=ncols,
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        nodata=nodata,
        compress='lzw'
    ) as dst:

        dst.write(raster, 1)

    print(f'Saved: {output_tif}')


In [ ]:
# ==========================
# Output directory
# ==========================

os.makedirs(
    '../../result/Soil',
    exist_ok=True
)


In [ ]:
# ==========================
# Export tif
# ==========================

datasets = {
    'maize': maize_env,
    'wheat': wheat_env
}

variables = ['pH', 'Sand', 'Silt', 'Clay']

for crop, df in datasets.items():

    for var in variables:

        output_tif = (
            f'../../result/soil_tif/'
            f'{crop}_{var}.tif'
        )
        points_to_raster(
            df=df,
            value_col=var,
            output_tif=output_tif,
            resolution=0.05
        )


pH
Grid cells: 94,868
Raster size: 682 × 1196
Saved: ../result/soil_tif/maize_pH.tif

Sand
Grid cells: 94,868
Raster size: 682 × 1196
Saved: ../result/soil_tif/maize_Sand.tif

Silt
Grid cells: 94,868
Raster size: 682 × 1196
Saved: ../result/soil_tif/maize_Silt.tif

Clay
Grid cells: 94,868
Raster size: 682 × 1196
Saved: ../result/soil_tif/maize_Clay.tif

pH
Grid cells: 29,817
Raster size: 702 × 1206
Saved: ../result/soil_tif/wheat_pH.tif

Sand
Grid cells: 29,817
Raster size: 702 × 1206
Saved: ../result/soil_tif/wheat_Sand.tif

Silt
Grid cells: 29,817
Raster size: 702 × 1206
Saved: ../result/soil_tif/wheat_Silt.tif

Clay
Grid cells: 29,817
Raster size: 702 × 1206
Saved: ../result/soil_tif/wheat_Clay.tif
